# Phase 6: Fine-tuning (QLoRA)
Run `notebooks/00_setup.ipynb` setup cell first, then start here.

## Session 1: prepare data and tiny test run

In [ ]:
%cd /content/llm-qa-project
!git fetch origin
!git checkout fine-tuning
!git pull

Build the balanced training set (4,000 train + 200 validation, 50% unanswerable, from the training split only).

In [ ]:
!python -m src.train_lora prepare

Tiny test run: 100 examples, 15 steps, about 5 to 10 minutes. Checks that training starts, loss goes down, and the adapter saves. The test adapter goes to `checkpoints/` and is NOT committed.

In [ ]:
!python -m src.train_lora train --tiny

Save Session 1 results (use your own name and GitHub email).

In [ ]:
!git config user.name "YOUR NAME"
!git config user.email "YOUR_GITHUB_EMAIL"
!git add results/finetune_data_stats.json results/finetune_tiny_log.json
!git commit -m "Phase 6 session 1: training data and tiny test run"
!git push origin fine-tuning

## Session 2: full training (about 1 to 1.5 hours)
After setup, run the checkout cell above, then `prepare` again (data files are not kept between sessions; same seed gives identical data).

In [ ]:
!python -m src.train_lora prepare
!python -m src.train_lora train

If it crashes but the server is still alive (e.g. out of memory), continue from the last checkpoint: `!python -m src.train_lora train --resume` (add `--batch_size 2 --grad_accum 8` for memory errors). If the server is lost, checkpoints are lost too; start the run again.

Save the adapter to GitHub right away. `-f` is needed because `.gitignore` blocks model files.

In [ ]:
!git config user.name "YOUR NAME"
!git config user.email "YOUR_GITHUB_EMAIL"
!git add -f adapters/phi3-squad-lora/
!git add results/finetune_train_log.json
!git commit -m "Phase 6 session 2: trained LoRA adapter and training log"
!git push origin fine-tuning

## Session 3: evaluate
After setup, run the checkout cell above (it pulls the saved adapter).

In [ ]:
!python -m src.eval_finetuned --limit 20

In [ ]:
!python -m src.eval_finetuned

In [ ]:
!python -m src.compare_results

In [ ]:
!git config user.name "YOUR NAME"
!git config user.email "YOUR_GITHUB_EMAIL"
!git add results/
!git commit -m "Phase 6 session 3: fine-tuned model evaluation"
!git push origin fine-tuning

## Session 4: depth experiments (Dr. Lu: go deep on one method)
After setup + the checkout cell. About 75 min of GPU for the training-size curve, about 70 more for the optional rank comparison. Each run is resumable: if the server dies, rerun the same cell.

**Training-size curve:** the same recipe on 500, 1,000 and 2,000 examples (the main run is the 4,000 point).

In [ ]:
!python -m src.train_lora prepare
for n in [500, 1000, 2000]:
    !python -m src.train_lora train --max_train {n} --name size{n}
    !python -m src.eval_finetuned --adapter adapters/size{n} --name finetuned_size{n}

**Optional: LoRA rank comparison** (rank 8 vs the main run's rank 16).

In [ ]:
!python -m src.train_lora train --rank 8 --name rank8
!python -m src.eval_finetuned --adapter adapters/rank8 --name finetuned_rank8

Save logs and scores (the extra adapters are ~50 MB each and are not committed; they can be retrained from the logged settings).

In [ ]:
!python -m src.compare_results
!git config user.name "YOUR NAME"
!git config user.email "YOUR_GITHUB_EMAIL"
!git add results/
!git commit -m "Phase 6 session 4: training-size curve and rank comparison"
!git push origin fine-tuning